In [ ]:
import os
from dotenv import load_dotenv
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import google.generativeai as genai

# Load environment variables
load_dotenv()


: 

In [ ]:


# Get API keys securely
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY not found. Check your .env file.")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found. Check your .env file.")


In [ ]:
def load_pdf_file(file_path):
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    return documents

pdf_path = "C:/Bits_hyd/doctor_pov/CuraMateDR/Data/tb.pdf"
extracted_data = load_pdf_file(pdf_path)


Number of Pages Loaded: 1874


In [ ]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

text_chunks = text_split(extracted_data)


Number of Text Chunks: 18109


In [ ]:
def get_hugging_face_embedding():
    return HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

embeddings = get_hugging_face_embedding()
print("Embeddings Loaded Successfully")


Embeddings Loaded Successfully


In [ ]:


# Initialize Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY, environment="us-east-1")

# Connect to the existing index
index_name = "medibot"
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name, 
    embedding=embeddings
)

Index is ready. Details: {'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 36218}},
 'total_vector_count': 36218}


In [ ]:
from langchain_pinecone import PineconeVectorStore

# Connect to the existing index
docsearch = PineconeVectorStore.from_existing_index(
    index_name="medibot", 
    embedding=embeddings
)

# Create retriever
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 4})
print("Retriever Created Successfully")


Retriever Created Successfully


In [ ]:
def retrieve_text(question):
    retrieved_docs = retriever.invoke(question)
    retrieved_texts = [doc.page_content for doc in retrieved_docs]
    return "\n".join(retrieved_texts)

# Example
query = "What are the symptoms of tuberculosis?"
retrieved_text = retrieve_text(query)
print("Retrieved Context:", retrieved_text)


NameError: name 'retriever' is not defined